# 组合优化——基本框架

组合优化本身并不引入新的资产，也没有对收益率的预测产生更新的观点，它的核心是**投资的分散化和风险的控制**。最早的组合构建可以追溯到上世纪 50 年代的 Markowitz 均值方差理论。

QuantStudio 的组合优化功能位于 `QuantStudio.PortfolioConstructor` 子模块中，提供以下核心组件：

| 组件 | 类型 | 功能 |
|------|------|------|
| `OptimizationObjective` | 抽象基类 | 优化目标基类，定义 `genObjective()` 接口 |
| `MeanVarianceObjective` | `OptimizationObjective` | 均值方差优化目标 |
| `RiskBudgetObjective` | `OptimizationObjective` | 风险预算优化目标 |
| `MaxDiversificationObjective` | `OptimizationObjective` | 最大分散化优化目标 |
| `Constraint` | 抽象基类 | 约束条件基类，定义 `genConstraint()` 接口 |
| `BudgetConstraint` | `Constraint` | 预算约束（总权重约束） |
| `WeightConstraint` | `Constraint` | 权重约束（上下限） |
| `FactorExposeConstraint` | `Constraint` | 因子暴露约束 |
| `VolatilityConstraint` | `Constraint` | 波动率约束 |
| `TurnoverConstraint` | `Constraint` | 换手率约束 |
| `ExpectedReturnConstraint` | `Constraint` | 预期收益约束 |
| `NonZeroNumConstraint` | `Constraint` | 非零权重数目约束 |
| `PortfolioConstructor` | 抽象基类 | 组合构造器基类，定义 `solve()` 接口 |
| `CVXPC` | `PortfolioConstructor` | 基于 CVXPY 的凸优化组合构造器 |

## 架构设计

组合优化的整体流程如下：

```
用户输入
  ├── mask (股票池)
  ├── expected_return (预期收益)
  ├── cov / factor_cov + factor_data + specific_risk (风险模型)
  ├── p0 (初始组合)
  └── bmk (基准组合)
        │
        ▼
  OptimizationObjective.genObjective()  →  {'type': 'Quadratic', 'Sigma': ..., 'Mu': ...}
        │
        ▼
  Constraint.genConstraint() × N       →  [{'type': 'Box', ...}, {'type': 'LinearEq', ...}, ...]
        │
        ▼
  PortfolioConstructor.solve()         →  (array(n) 权重, dict 求解信息)
        │
        ▼
  CVXPC._solve()                       →  cvxpy 建模 + 求解器求解
```

**核心设计模式**：
- **目标-约束分离**：优化目标与约束条件是独立的对象，可自由组合
- **数学形式标准化**：`genObjective()` 和 `genConstraint()` 产出标准化的数学描述（dict），解耦了业务逻辑和求解器
- **约束松弛机制**：约束条件支持 `DropPriority` 优先级，当求解失败时按优先级自动松弛约束

## 优化目标

### OptimizationObjective — 优化目标基类

所有优化目标都继承自 `OptimizationObjective`，需要实现 `genObjective()` 方法。该方法返回一个字典，描述优化目标的数学形式：

| 类型 | 返回格式 | 说明 |
|------|----------|------|
| `Linear` | `{'f': array(n,), 'type': 'Linear'}` | 线性目标：$\mathbf{f}^T\mathbf{x}$ |
| `Quadratic` | `{'Sigma': array(n,n), 'Mu': array(n,), 'type': 'Quadratic'}` | 二次目标：$\mathbf{x}^T\Sigma\mathbf{x} + \boldsymbol{\mu}^T\mathbf{x}$ |
| `Quadratic`（因子模型） | `{'X': array(n,k), 'F': array(k,k), 'Delta': array(n,), 'Mu': array(n,), 'type': 'Quadratic'}` | 二次目标（因子形式）：$\mathbf{x}^T(XFX^T+\Delta)\mathbf{x} + \boldsymbol{\mu}^T\mathbf{x}$ |
| `L1_Linear` | `Linear` + L1 惩罚项 | 含交易成本惩罚的线性目标 |
| `L1_Quadratic` | `Quadratic` + L1 惩罚项 | 含交易成本惩罚的二次目标 |
| `Risk_Budget` | `{'Sigma': ..., 'b': array(n,), 'type': 'Risk_Budget'}` | 风险预算目标 |
| `Max_Diversification` | `{'Sigma': ..., 'type': 'Max_Diversification'}` | 最大分散化目标 |
| `Sharpe` | `{'f': ..., 'Sigma': ..., 'type': 'Sharpe'}` | 最大夏普率目标 |

每个目标都包含 `minmax` 字段（`'min'` 或 `'max'`），表示最小化还是最大化。

### MeanVarianceObjective — 均值方差优化目标

Markowitz 均值方差模型：在给定约束条件下期望效用最大化。优化问题的一般形式如下：

$$
\begin{aligned}
  & \underset{\mathbf{w}}{\mathop{\max }}\,\left\{\gamma\cdot\mathbf{\mu}^T\cdot(\mathbf{w}-\mathbf{w}_b)-\frac{\lambda }{2}(\mathbf{w}-\mathbf{w}_b)^T\mathbf{\Sigma}(\mathbf{w}-\mathbf{w}_b)-\operatorname{TC}\left( \mathbf{w} \right) \right\} \\ 
 & \operatorname{TC}\left( \mathbf{w} \right)={\lambda_1}\sum\limits_{i=1}^n{\left| {{w}_{i}}-{{w}_{0i}} \right|} + {\lambda_2}\sum\limits_{i=1}^n{\left( {{w}_{i}}-{{w}_{0i}} \right)^{+}} + {\lambda_3}\sum\limits_{i=1}^n{{\left( {{w}_{i}}-{{w}_{0i}} \right)}^{-}} \\
 & s.t.\ \mathbf{w}\in\mathfrak{C}
\end{aligned}
$$

**参数列表**（通过 `args` 传入）：

| 参数 | 类型 | 默认值 | 说明 |
|------|------|--------|------|
| `Benchmark` | `bool` | `False` | 是否相对基准优化，True 时优化变量为 $\mathbf{w}-\mathbf{w}_b$ |
| `ExpectedReturnCoef` | `float` | `0.0` | 收益项系数 $\gamma$，0 表示纯风险最小化 |
| `RiskAversionCoef` | `float` | `1.0` | 风险厌恶系数 $\lambda$ |
| `TurnoverPenaltyCoef` | `float` | `0.0` | 双边换手惩罚系数 $\lambda_1$ |
| `BuyPenaltyCoef` | `float` | `0.0` | 买入惩罚系数 $\lambda_2$ |
| `SellPenaltyCoef` | `float` | `0.0` | 卖出惩罚系数 $\lambda_3$ |

**构造方法**：

```python
MeanVarianceObjective(
    mask,                    # array(shape=(n,)), 股票池，True 表示可选
    expected_return=None,    # array(shape=(n,)), 预期收益
    p0=None,                 # array(shape=(n,)), 初始组合（换手惩罚时需要）
    bmk=None,                # array(shape=(n,)), 基准组合（相对基准时需要）
    factor_cov=None,         # array(shape=(k,k)), 因子协方差阵
    factor_data=None,        # array(shape=(n,k)), 因子暴露矩阵
    specific_risk=None,      # array(shape=(n,)), 特异性风险
    cov=None,                # array(shape=(n,n)), 证券协方差阵
    args={},                 # 参数设置
    config_file=None         # 配置文件路径
)
```

> **风险模型输入**：支持两种方式——① 直接传入 `cov`（证券协方差阵）；② 传入 `factor_cov` + `factor_data` + `specific_risk`（因子模型形式）。如果三者均非 None，优先使用因子模型计算协方差阵：$\Sigma = XFX^T + \Delta$。

### RiskBudgetObjective — 风险预算优化目标

详见 [风险预算模型](./风险预算模型.ipynb)。

**参数列表**（通过 `args` 传入）：无额外参数。

```python
RiskBudgetObjective(
    mask,                # array(shape=(n,)), 股票池
    budget=None,         # array(shape=(n,)), 风险预算，None 表示等风险预算（风险平价）
    factor_cov=None,     # 因子协方差阵
    factor_data=None,    # 因子暴露矩阵
    specific_risk=None,  # 特异性风险
    cov=None,            # 证券协方差阵
    args={},
    config_file=None
)
```

## 约束条件

大多数组合构建最终都归结为满足一定约束条件的最优化问题。不同的组合构建问题的优化目标各不一样，但常见的约束条件基本属于以下范畴。

所有约束条件都继承自 `Constraint`，需要实现 `genConstraint()` 方法，返回标准化约束列表。每个约束都有 `DropPriority` 参数（默认 -1，表示不可松弛），当求解失败时按优先级从高到低（数值从大到小）依次松弛。

### 约束条件的数学形式

| 类型 | 返回格式 | 对应数学形式 |
|------|----------|-------------|
| `Box` | `{'lb': array(n,), 'ub': array(n,), 'type': 'Box'}` | $\mathbf{lb} \le \mathbf{x} \le \mathbf{ub}$ |
| `LinearIn` | `{'A': array(m,n), 'b': array(m,), 'type': 'LinearIn'}` | $A\mathbf{x} \le \mathbf{b}$ |
| `LinearEq` | `{'Aeq': array(m,n), 'beq': array(m,), 'type': 'LinearEq'}` | $A_{eq}\mathbf{x} = \mathbf{b}_{eq}$ |
| `Quadratic` | `{'Sigma': array(n,n), 'Mu': array(n,), 'q': float, 'type': 'Quadratic'}` | $\mathbf{x}^T\Sigma\mathbf{x} + \boldsymbol{\mu}^T\mathbf{x} \le q$ |
| `L1` | `{'c': array(n,), 'l': float, 'type': 'L1'}` | $\sum \|x_i - c_i\| \le l$ |
| `Pos` | `{'c_pos': array(n,), 'l_pos': float, 'type': 'Pos'}` | $\sum(x_i - c_i)^+ \le l_{pos}$ |
| `Neg` | `{'c_neg': array(n,), 'l_neg': float, 'type': 'Neg'}` | $\sum(x_i - c_i)^- \le l_{neg}$ |
| `NonZeroNum` | `{'b': array(n,), 'N': float, 'type': 'NonZeroNum'}` | $\sum\mathbb{I}(x_i \neq 0) \le N$ |

### BudgetConstraint — 预算约束

限制组合的总权重。对应数学形式：$\mathbf{1}^T(\mathbf{w}-\mathbf{w}_b) \in [a_{down}, a_{up}]$

**参数列表**（通过 `args` 传入）：

| 参数 | 类型 | 默认值 | 说明 |
|------|------|--------|------|
| `UpLimit` | `float` | `1.0` | 总权重上限 |
| `DownLimit` | `float` | `1.0` | 总权重下限 |
| `Benchmark` | `bool` | `False` | 是否相对基准 |

```python
# 全额投资约束（权重之和 = 1）
BudgetConstraint(mask=Mask, args={"UpLimit": 1, "DownLimit": 1})

# 相对基准的预算约束
BudgetConstraint(mask=Mask, bmk=Bmk, args={"UpLimit": 0.05, "DownLimit": -0.05, "Benchmark": True})
```

### WeightConstraint — 权重约束

限制单个证券的权重范围。对应数学形式：$\mathbf{a} \le \mathbf{w}-\mathbf{w}_b \le \mathbf{b}$

**参数列表**（通过 `args` 传入）：

| 参数 | 类型 | 默认值 | 说明 |
|------|------|--------|------|
| `Benchmark` | `bool` | `False` | 是否相对基准 |

```python
# 纯多头约束（权重在 [0, 1] 之间）
WeightConstraint(mask=Mask, up_limit=1, down_limit=0)

# 个股权重上限（可为标量或向量）
WeightConstraint(mask=Mask, up_limit=0.1)

# 相对基准的权重偏离约束
WeightConstraint(mask=Mask, bmk=Bmk, up_limit=0.02, down_limit=-0.02, args={"Benchmark": True})
```

### FactorExposeConstraint — 因子暴露约束

限制组合相对于基准的因子暴露。对应数学形式：$\mathbf{x}^T(\mathbf{w}-\mathbf{w}_b) \in [a_{down}, a_{up}]$

**参数列表**（通过 `args` 传入）：

| 参数 | 类型 | 默认值 | 说明 |
|------|------|--------|------|
| `FactorType` | `Literal["数值型", "类别型"]` | `"数值型"` | 因子类型，类别型会自动转换为哑变量 |
| `UpLimit` | `float` | `1.0` | 因子暴露上限 |
| `DownLimit` | `float` | `1.0` | 因子暴露下限 |
| `Benchmark` | `bool` | `False` | 是否相对基准 |

```python
# 风格中性约束（行业因子暴露与基准一致）
FactorExposeConstraint(
    mask=Mask, factor_data=IndustryDummy,
    bmk=Bmk,
    args={"UpLimit": 0, "DownLimit": 0, "Benchmark": True, "FactorType": "类别型"}
)
```

### VolatilityConstraint — 波动率约束

限制组合的波动率（或跟踪误差）。对应数学形式：$(\mathbf{w}-\mathbf{w}_b)^T\Sigma(\mathbf{w}-\mathbf{w}_b) \le \sigma^2$

**参数列表**（通过 `args` 传入）：

| 参数 | 类型 | 默认值 | 说明 |
|------|------|--------|------|
| `UpLimit` | `float` | `0.06` | 波动率上限（年化） |
| `Benchmark` | `bool` | `False` | 是否相对基准（为 True 时限制的是跟踪误差） |

```python
# 组合年化波动率不超过 10%
VolatilityConstraint(mask=Mask, cov=Cov, args={"UpLimit": 0.10})

# 跟踪误差不超过 3%
VolatilityConstraint(mask=Mask, bmk=Bmk, cov=Cov, args={"UpLimit": 0.03, "Benchmark": True})
```

### TurnoverConstraint — 换手率约束

限制组合换手率。可以限制总换手、总买入、总卖出、个券买卖等。

**参数列表**（通过 `args` 传入）：

| 参数 | 类型 | 默认值 | 说明 |
|------|------|--------|------|
| `ConstraintType` | `Literal["总换手限制", "总买入限制", "总卖出限制", "买卖限制", "买入限制", "卖出限制"]` | `"总换手限制"` | 限制类型 |
| `AmtMultiple` | `float` | `1.0` | 成交额倍数（个券限制时使用） |
| `UpLimit` | `float` | `0.7` | 限制上限 |

```python
# 总换手不超过 50%
TurnoverConstraint(mask=Mask, p0=P0, args={"ConstraintType": "总换手限制", "UpLimit": 0.5})

# 总买入不超过 30%（卖出不受限）
TurnoverConstraint(mask=Mask, p0=P0, args={"ConstraintType": "总买入限制", "UpLimit": 0.3})

# 个券买卖不超过日均成交额的 10%
TurnoverConstraint(mask=Mask, p0=P0, wealth=1e6, amt=DailyAmt, args={"ConstraintType": "买卖限制", "AmtMultiple": 0.1})
```

### ExpectedReturnConstraint — 预期收益约束

限制组合的预期收益下限。对应数学形式：$\boldsymbol{\mu}^T(\mathbf{w}-\mathbf{w}_b) \ge a$

**参数列表**（通过 `args` 传入）：

| 参数 | 类型 | 默认值 | 说明 |
|------|------|--------|------|
| `DownLimit` | `float` | `0.0` | 预期收益下限 |
| `Benchmark` | `bool` | `False` | 是否相对基准 |

```python
# 组合预期收益不低于 5%（年化）
ExpectedReturnConstraint(mask=Mask, expected_return=ExpectedReturn, args={"DownLimit": 0.05})
```

### NonZeroNumConstraint — 非零权重数目约束

限制组合中持仓数量。对应数学形式：$\operatorname{nnz}(\mathbf{w}-\mathbf{w}_b) \le N$

> **注意**：该约束引入了 0-1 整数变量，会将问题转换为混合整数规划（MIP），求解难度和时间大幅增加。

**参数列表**（通过 `args` 传入）：

| 参数 | 类型 | 默认值 | 说明 |
|------|------|--------|------|
| `UpLimit` | `int` | `150` | 持仓数量上限 |
| `Benchmark` | `bool` | `False` | 是否相对基准 |

```python
# 持仓不超过 50 只
NonZeroNumConstraint(mask=Mask, args={"UpLimit": 50})
```

## CVXPC — 组合构造器

`CVXPC` 是基于 CVXPY 的凸优化组合构造器，继承自 `PortfolioConstructor`。它接收一个优化目标和一组约束条件，通过 `solve()` 方法求解最优组合权重。

### 构造参数

| 参数 | 类型 | 默认值 | 说明 |
|------|------|--------|------|
| `OptimOption` | `dict` | `{}` | 优化选项，传递给 `cvxpy.Problem.solve(**OptimOption)` |

### solve() 方法

```python
PC = CVXPC(mask=Mask, objective=Objective, constraints=ConstraintList, args={"OptimOption": {...}})
Portfolio, Info = PC.solve()
```

**返回值**：
- `Portfolio`：`array(shape=(n,))`，完整长度（与原始 mask 长度一致）的最优权重向量，不在股票池中的 ID 对应位置为 NaN
- `Info`：`dict`，求解信息：
  - `status`：1 表示求解成功，0 表示失败
  - `msg`：求解器返回的状态信息
  - `solver_name`：使用的求解器名称
  - `solve_time`：求解耗时（秒）
  - `setup_time`：问题构建耗时（秒）
  - `num_iters`：求解迭代次数
  - `ReleasedConstraint`：被松弛的约束条件列表

### 求解器选择

通过 `OptimOption` 中的 `solver` 参数选择求解器：

| 求解器 | 适用场景 | 特点 |
|--------|----------|------|
| `cvx.SCIP` | MIP 问题（含 NonZeroNum 约束） | 开源，支持整数规划 |
| `cvx.CLARABEL` | 锥规划（RiskBudget 等） | 开源，支持指数锥 |
| `cvx.OSQP` | 二次规划 | 开源，ADMM 算子分裂 |
| `cvx.ECOS` | 锥规划 | 开源，内点法 |

```python
# 使用 OSQP 求解器
PC = CVXPC(mask=Mask, objective=Objective, constraints=ConstraintList, 
           args={"OptimOption": {"solver": cvx.OSQP, "verbose": False}})

# 使用 SCIP 求解器（含整数约束时推荐）
PC = CVXPC(mask=Mask, objective=Objective, constraints=ConstraintList,
           args={"OptimOption": {"solver": cvx.SCIP, "verbose": True}})
```

## 完整示例

以下展示一个完整的最小方差组合求解示例：

In [ ]:
import numpy as np
import cvxpy as cvx

# DEMO 数据
np.random.seed(0)
nID = 10
Mask = np.full(shape=(nID,), fill_value=True, dtype=np.bool)
ExpectedReturn = np.random.randn(nID)
Cov = np.cov(np.random.randn(100 * nID, nID), rowvar=False)
P0 = np.random.rand(nID)
P0 = P0 / np.sum(P0)
Bmk = np.random.rand(nID)
Bmk = Bmk / np.sum(Bmk)

In [ ]:
# 最小方差组合
from QuantStudio.PortfolioConstructor.CVXPC import CVXPC
from QuantStudio.PortfolioConstructor.BasePC import MeanVarianceObjective, BudgetConstraint, WeightConstraint

Objective = MeanVarianceObjective(
    mask=Mask, expected_return=ExpectedReturn, p0=P0, cov=Cov,
    args={"ExpectedReturnCoef": 0, "RiskAversionCoef": 1}
)
ConstraintList = [
    WeightConstraint(mask=Mask, up_limit=1, down_limit=0),
    BudgetConstraint(mask=Mask, args={"UpLimit": 1, "DownLimit": 1})
]

PC = CVXPC(mask=Mask, objective=Objective, constraints=ConstraintList,
           args={"OptimOption": {"solver": cvx.SCIP, "verbose": True}})
Portfolio, Info = PC.solve()

print(np.where(np.abs(Portfolio) > 1e-8, Portfolio, 0))
print(Info)

## 模型对比与选择

| 模型 | 数学形式 | 输入要求 | 特点 | 适用场景 |
|------|----------|----------|------|----------|
| 最小方差 | $\min \mathbf{w}^T\Sigma\mathbf{w}$ | 仅需协方差阵 | 对参数估计敏感度较低，但持仓可能集中 | 纯风险控制 |
| 均值方差 | $\max \gamma\boldsymbol{\mu}^T\mathbf{w} - \frac{\lambda}{2}\mathbf{w}^T\Sigma\mathbf{w}$ | 预期收益 + 协方差阵 | 收益估计误差影响大，权重对输入敏感 | 收益-风险权衡 |
| 风险预算 | $\min \sum(\mathcal{RC}_i - b_i\mathcal{R})^2$ | 协方差阵 + 风险预算 | 风险分散化，权重较均衡 | 风险分散化投资 |

详见各专题文档：
- [均值方差模型](./均值方差模型.ipynb)
- [风险预算模型](./风险预算模型.ipynb)
- [组合优化策略](./组合优化策略.ipynb)